In [17]:
# ライブラリのインポート

import pandas as pd
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report


In [ ]:
# データの読み込み

# 学習データの読み込み
df_train = pd.read_csv("data/train.csv")
# テストデータの読み込み
df_test = pd.read_csv("data/test.csv")

new_col_name = ["id", "生存結果", "客室のクラス", "性別", "年齢", "兄弟・配偶者の数", "両親・子供の数", "運賃", "乗船した港"]
# 列名を変更
df_train.columns = new_col_name

df_test.columns = ["id", "客室のクラス", "性別", "年齢", "兄弟・配偶者の数", "両親・子供の数", "運賃", "乗船した港"]

# LightGBMのためにカテゴリ変数に変換
cat_cols = ["性別", "乗船した港"]
for c in cat_cols:
    df_train[c] = df_train[c].astype("category")
    df_test[c] = df_test[c].astype("category")

print(df_train.head())
print(df_test.head())



   id  生存結果  客室のクラス      性別    年齢  兄弟・配偶者の数  両親・子供の数       運賃 乗船した港
0   3     1       1  female  35.0         1        0  53.1000     S
1   4     0       3    male  35.0         0        0   8.0500     S
2   7     0       3    male   2.0         3        1  21.0750     S
3   9     1       2  female  14.0         1        0  30.0708     C
4  11     1       1  female  58.0         0        0  26.5500     S
   id  客室のクラス      性別    年齢  兄弟・配偶者の数  両親・子供の数       運賃 乗船した港
0   0       3    male  22.0         1        0   7.2500     S
1   1       1  female  38.0         1        0  71.2833     C
2   2       3  female  26.0         0        0   7.9250     S
3   5       3    male   NaN         0        0   8.4583     Q
4   6       1    male  54.0         0        0  51.8625     S


In [ ]:
# モデルの学習＆評価

# 学習データ：説明変数と目的変数の作成
X_base = df_train.drop("生存結果", axis=1)
y_base = df_train["生存結果"]

# データの分割
X_train, X_test, y_train, y_test = train_test_split(X_base, y_base, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape, X_base.shape, y_base.shape)

# LightGBMようのデータセット作成
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_test = lgb.Dataset(X_test, label=y_test, reference=lgb_train)

# 評価基準のパラメータを作成する
# →Claudeに相談して決めたパラメータ。機械学習の種類によって違いそうなのはなんとなく理解できる。
params = {
    "objective": "binary",        # 2値分類
    "metric": "binary_logloss",   # 分類用の評価指標
    "verbosity": -1,
    "seed": 42,
}
# 学習データからモデルを作る
# →Claudeに書いてもらったコード。あまり調べていないので、パラメータと学習データを引き渡すことしか理解していない。
gbm = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_test],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50),  # 改善しなくなったら打ち切り
        lgb.log_evaluation(period=50),
    ],
)
# モデルの評価
y_pred_proba = gbm.predict(X_test, num_iteration=gbm.best_iteration)
y_pred = (y_pred_proba > 0.5).astype(int) # いったんしきい値を0.5にする。それ以外も試したが、あまり精度は上がらなかった。
print(y_pred)
print(classification_report(y_test, y_pred))

y_result_proba = gbm.predict(df_test, num_iteration=gbm.best_iteration)
y_result = (y_result_proba > 0.5).astype(int)# いったんしきい値を0.5にする。それ以外も試したが、あまり精度は上がらなかった。
print(y_result)

# 5. 予測結果の確認
print(f"\n生存予測: {y_result.sum()}人")
print(f"死亡予測: {(y_result == 0).sum()}人")

# 提出ファイル作成
submission_rf = pd.DataFrame({
    'id': df_test['id'],
    'Survived': y_result
})
print(submission_rf.head())

submission_rf.to_csv('nsubmission_gbm.csv', header=False, index=False)
print("\nsubmission_gbm.csv を保存しました!")



(356, 8) (89, 8) (356,) (89,) (445, 8) (445,)
Training until validation scores don't improve for 50 rounds
[50]	training's binary_logloss: 0.232043	valid_1's binary_logloss: 0.368548
Early stopping, best iteration is:
[36]	training's binary_logloss: 0.283066	valid_1's binary_logloss: 0.353477
[0 1 1 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 1 0 1 0 0 1 1 0 0 0 1 0 0 1 0 1 0
 0 1 0 1 0 0 0 1 0 0 1 0 0 1 0 1 0 0 1 0 0 0 1 1 1 1 0 0 1 0 0 0 0 1 0 0 0
 0 0 1 0 1 1 0 1 0 1 0 0 1 1 1]
              precision    recall  f1-score   support

           0       0.89      0.89      0.89        56
           1       0.82      0.82      0.82        33

    accuracy                           0.87        89
   macro avg       0.86      0.86      0.86        89
weighted avg       0.87      0.87      0.87        89

[0 1 1 0 0 0 1 0 1 1 0 1 0 0 1 0 0 1 0 0 0 1 1 0 0 0 0 1 0 0 1 0 0 0 0 0 0
 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 1 1 1 1 0
 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 